In [1]:
import os

import numpy as np
import pandas as pd

from pyscenic.cli.utils import (
    ATTRIBUTE_NAME_CELL_IDENTIFIER,
    ATTRIBUTE_NAME_GENE,
    append_auc_mtx,
    is_valid_suffix,
    load_adjacencies,
    load_exp_matrix,
    load_modules,
    load_signatures,
    save_enriched_motifs,
    save_matrix,
    suffixes_to_separator,
)

In [ ]:
from pyscenic.utils import add_correlation, modules_from_adjacencies

In [89]:
pdir = "/dcs04/lieber/marmaypag/spatialDLPFC_mdd_bpd_LIBD4100/spatialDLPFC_mdd_bpd/"
RESULTS_DIR = pdir+"processed-data/09_SCENIC/"
DATASET_ID="spe-n119_21077"

In [3]:
adjacencies = load_adjacencies(pdir+"processed-data/09_SCENIC/spe-n119_21077_adj_with-corr.csv")
adjacencies.shape

(1312884, 5)

In [4]:
adjacencies.head()

,TF,target,importance,regulation,rho
0,PRNP,GPM6A,41.904825,1,0.6890393220321283
1,PRNP,CHN1,39.875782,1,0.7033438801318207
2,MYRF,PLP1,38.896794,1,0.5970946738999354
3,PRNP,CALM1,38.549526,1,0.7179264885218953
4,PRNP,CPE,37.955888,1,0.6147579443825747


In [5]:
laDEGs = pd.read_csv(pdir+"processed-data/07_dx_DE/layer-adjusted-pc3-age-nspots_smoothed-k9-1663_dx-sex_degs-F-test-t-test.csv")
lrDEGs = pd.read_csv(pdir+"processed-data/07_dx_DE/layer-restricted-pc3-age-nspots_smoothed-k9-1663_dx-sex_degs-F-test-t-test.csv")

In [39]:
mbvDEGs = list(set(pd.concat([laDEGs.gene_name, lrDEGs.gene_name])))
len(mbvDEGs)

649

In [93]:
print("Number of MBv F sig. genes:", len(mbvDEGs))

Number of MBv F sig. genes: 649


In [7]:
#mbvDEGs = laDEGs[laDEGs.n_ttest_sig>0].gene_name
#len(mbvDEGs)

293

In [46]:
adjacencies1 = adjacencies[adjacencies.target.isin(mbvDEGs)]
adjacencies1.shape

(34164, 5)

In [47]:
adjacencies2 = adjacencies[adjacencies.TF.isin(mbvDEGs)]
adjacencies2.shape

(105796, 5)

In [10]:
from functools import partial
from itertools import chain
from typing import Sequence, Type
from urllib.parse import urljoin

import numpy as np
import pandas as pd
from ctxcore.genesig import GeneSignature, Regulon, openfile

from pyscenic.utils import modules4thr, modules4top_targets, modules4top_factors, _create_idx_pairs

COLUMN_NAME_TF = "TF"
COLUMN_NAME_MOTIF_ID = "MotifID"
COLUMN_NAME_MOTIF_SIMILARITY_QVALUE = "MotifSimilarityQvalue"
COLUMN_NAME_ORTHOLOGOUS_IDENTITY = "OrthologousIdentity"
COLUMN_NAME_ANNOTATION = "Annotation"

COLUMN_NAME_TARGET = "target"
COLUMN_NAME_WEIGHT = "importance"
COLUMN_NAME_REGULATION = "regulation"
COLUMN_NAME_CORRELATION = "rho"
RHO_THRESHOLD = 0.03

In [11]:
def select_modules_from_adjacencies(
    adjacencies: pd.DataFrame,
#    ex_mtx: pd.DataFrame,
    thresholds=(0.75, 0.90),
    top_n_targets=(50,),
    top_n_regulators=(5, 10, 50),
    min_genes=20,
    absolute_thresholds=True,
    keep_only_activating=False
) -> Sequence[Regulon]:
    # To make the pySCENIC code more robust to the selection of the network inference method in the first step of
    # the pipeline, it is better to use percentiles instead of absolute values for the weight thresholds.
    if not absolute_thresholds:

        def iter_modules(adjc, context):
            yield from chain(
                chain.from_iterable(
                    modules4thr(
                        adjc, thr, context, pattern="weight>{}%".format(frac * 100)
                    )
                    for thr, frac in zip(
                        list(adjacencies["importance"].quantile(thresholds)),
                        thresholds,
                    )
                ),
#                chain.from_iterable(
#                    modules4top_targets(adjc, n, context) for n in top_n_targets
#                ),
#                chain.from_iterable(
#                    modules4top_factors(adjc, n, context) for n in top_n_regulators
#                ),
            )

    else:

        def iter_modules(adjc, context):
            yield from chain(
                chain.from_iterable(
                    modules4thr(adjc, thr, context) for thr in thresholds
                ),
#                chain.from_iterable(
#                    modules4top_targets(adjc, n, context) for n in top_n_targets
#                ),
#                chain.from_iterable(
#                    modules4top_factors(adjc, n, context) for n in top_n_regulators
#                ),
            )
    
    
    activating_modules = adjacencies[adjacencies['regulation'] > 0.0]
    if keep_only_activating:
        modules_iter = iter_modules(
            activating_modules, frozenset(['activating'])
        )
    else:
        repressing_modules = adjacencies[adjacencies['regulation'] < 0.0]
        modules_iter = chain(
            iter_modules(activating_modules, frozenset(['activating'])),
            iter_modules(repressing_modules, frozenset(['repressing'])),
        )

    # Derive modules for these adjacencies.
    # + Add the transcription factor to the module.
    #   [We are unable to assess if a TF works in a direct self-regulating way, either inhibiting its own expression or
    #    activating it. Therefore the most unbiased way forward is to add the TF to both activating as well as
    #    repressing modules]
    # + Filter for minimum number of genes.
    #LOGGER.info("Creating modules.")

    def add_tf(module):
        return module.add(module.transcription_factor)

    return list(filter(lambda m: len(m) >= min_genes, map(add_tf, modules_iter)))


In [39]:
type(adjacencies2)

pandas.core.frame.DataFrame

In [56]:
modules = select_modules_from_adjacencies(adjacencies, thresholds=(0.5,), min_genes=10)

In [42]:
modules2 = select_modules_from_adjacencies(adjacencies2, thresholds=(0.5,))

In [57]:
mod_df = pd.DataFrame({"TF": [item.transcription_factor for item in modules], 
                       "dir": ["activating" if "activating" in item.context else "repressing" for item in modules],
                       "set_size": [len(item) for item in modules],
                      "n_DEGs": [len(set(item.genes).intersection(set(mbvDEGs))) for item in modules]})

In [58]:
mod_df.shape

(2010, 4)

In [67]:
mod_df[mod_df.TF=="ELK1"]

,TF,dir,set_size,n_DEGs
158,ELK1,activating,51,1
1238,ELK1,repressing,26,1


In [69]:
def collapse(lst):
    return '/' .join(chain.from_iterable(lst))

In [73]:
collapse(list(modules[1].genes))

'D/L/G/A/P/1/-/A/S/1/S/T/A/R/D/8/H/E/L/B/Z/S/W/I/M/8/-/A/S/1/K/I/F/1/8/A/R/N/F/1/5/7/-/A/S/1/C/9/o/r/f/5/0/L/C/N/1/2/A/C/0/9/3/6/9/0/./1/A/B/L/1/A/K/A/P/3/A/F/1/6/5/1/4/7/./1/M/S/H/4/A/C/1/0/4/3/3/5/./1/L/I/N/C/0/0/9/2/6/A/L/3/5/5/8/8/1/./1/A/C/0/1/2/3/6/8/./1/A/P/0/0/1/0/9/4/./2/N/I/D/2/N/C/A/P/H/A/L/3/9/2/1/7/2/./1/A/L/4/5/0/3/0/6/./1/P/A/R/P/1/2/A/D/A/A/C/0/7/9/2/0/9/./1/A/C/0/1/0/9/3/1/./3/A/L/1/3/8/7/6/2/./1/A/C/0/9/2/9/3/9/./1/C/Y/P/4/F/3/P/W/R/N/1/D/K/K/2'

In [74]:
mod_df pd.DataFrame({'/' .join(list(modules[1].genes))

'DLGAP1-AS1/STARD8/HELB/ZSWIM8-AS1/KIF18A/RNF157-AS1/C9orf50/LCN12/AC093690.1/ABL1/AKAP3/AF165147.1/MSH4/AC104335.1/LINC00926/AL355881.1/AC012368.1/AP001094.2/NID2/NCAPH/AL392172.1/AL450306.1/PARP12/ADA/AC079209.1/AC010931.3/AL138762.1/AC092939.1/CYP4F3/PWRN1/DKK2'

In [77]:
'/' .join(list(modules[1].genes))

'DLGAP1-AS1/STARD8/HELB/ZSWIM8-AS1/KIF18A/RNF157-AS1/C9orf50/LCN12/AC093690.1/ABL1/AKAP3/AF165147.1/MSH4/AC104335.1/LINC00926/AL355881.1/AC012368.1/AP001094.2/NID2/NCAPH/AL392172.1/AL450306.1/PARP12/ADA/AC079209.1/AC010931.3/AL138762.1/AC092939.1/CYP4F3/PWRN1/DKK2'

['ITPKB', 'ITGAX', 'PLP1', 'SREBF1', 'IRF1']

In [81]:
'/' .join(list(set(modules[646].genes).intersection(set(mbvDEGs))))

'ITPKB/ITGAX/PLP1/SREBF1/IRF1'

In [82]:
mod_df = pd.DataFrame({"TF": [item.transcription_factor for item in modules], 
                       "dir": ["activating" if "activating" in item.context else "repressing" for item in modules],
                       "set_size": [len(item) for item in modules],
                      "n_DEGs": [len(set(item.genes).intersection(set(mbvDEGs))) for item in modules],
                      "DEG_str": ['/' .join(list(set(item.genes).intersection(set(mbvDEGs)))) for item in modules]})

In [98]:
tmp = mod_df[mod_df.set_size<20]
#tmp[tmp.n_DEGs>5]
tmp.sort_values('n_DEGs')

,TF,dir,set_size,n_DEGs,DEG_str
1480,NR3C1,repressing,10,0,
1452,NFAT5,repressing,19,0,
1456,NFIB,repressing,16,0,
1458,NFIL3,repressing,15,0,
1459,NFKB1,repressing,13,0,
...,...,...,...,...,...
818,ZIK1,activating,13,2,SELPLG/FPR1
300,IRF1,activating,19,3,SOCS3/CCL2/IRF1
642,SP9,activating,13,3,CORT/GAD2/GAD1
220,GPD1,activating,19,3,TF/PLP1/RNASE1


In [99]:
tmp[tmp.TF.isin(mbvDEGs)]

,TF,dir,set_size,n_DEGs,DEG_str
19,ARG2,activating,17,1,ARG2
300,IRF1,activating,19,3,SOCS3/CCL2/IRF1
1111,AFF4,repressing,15,1,AFF4
1120,ARG2,repressing,14,1,ARG2
1122,ARID5B,repressing,16,1,ARID5B
1133,ATF3,repressing,13,1,ATF3
1146,BCL6,repressing,19,1,BCL6
1212,DLX1,repressing,19,1,DLX1
1230,EDN1,repressing,16,2,IL1R1/EDN1
1252,ETV5,repressing,15,1,ETV5


In [86]:
mod_df.to_csv(pdir+"processed-data/09_SCENIC/spe-n119_21077_modules.csv")

In [91]:
os.path.join(RESULTS_DIR, '{}_adj_with-corr.csv'.format(DATASET_ID))

'/dcs04/lieber/marmaypag/spatialDLPFC_mdd_bpd_LIBD4100/spatialDLPFC_mdd_bpd/processed-data/09_SCENIC/spe-n119_21077_adj_with-corr.csv'

In [90]:

REGULONS_DAT_FNAME = os.path.join(RESULTS_DIR, '{}.regulons.dat'.format(DATASET_ID))
REGULONS_DAT_FNAME

'/dcs04/lieber/marmaypag/spatialDLPFC_mdd_bpd_LIBD4100/spatialDLPFC_mdd_bpd/processed-data/09_SCENIC/spe-n119_21077.regulons.dat'

In [71]:
REGULONS_DAT_FNAME = .regulons.dat
with open(REGULONS_DAT_FNAME, 'wb') as f:
    pickle.dump(regulons, f)

tuple

In [63]:
mod_df2 = mod_df[mod_df.n_DEGs>=5]
mod_df2.shape

(122, 4)

In [65]:
mod_df2.sort_values('n_DEGs', ascending=False)

,TF,dir,set_size,n_DEGs
727,UBB,activating,12801,520
751,YWHAE,activating,12506,518
526,PRNP,activating,12775,517
162,ENO1,activating,11682,475
752,YWHAZ,activating,12325,465
...,...,...,...,...
646,SREBF1,activating,75,5
339,LARP1,activating,67,5
13,AIRE,activating,26,5
408,MYLK,activating,88,5


In [66]:
mod_df2[mod_df2.TF.isin(mbvDEGs)].sort_values('n_DEGs', ascending=False)

,TF,dir,set_size,n_DEGs
119,CYCS,activating,8805,339
523,PRDX5,activating,1593,137
579,RPS4X,activating,607,132
75,CEBPD,activating,306,84
578,RPS10,activating,341,68
184,FOS,activating,277,67
635,SOX9,activating,287,46
310,JUN,activating,187,30
345,LMO2,activating,118,26
680,TCF4,activating,155,19


In [60]:
mod_df[mod_df.TF.isin(mbvDEGs)].sort_values('set_size', ascending=False)

,TF,dir,set_size,n_DEGs
119,CYCS,activating,8805,339
523,PRDX5,activating,1593,137
579,RPS4X,activating,607,132
578,RPS10,activating,341,68
75,CEBPD,activating,306,84
...,...,...,...,...
1422,MNT,repressing,12,1
1777,ZIC1,repressing,11,1
1349,IRF1,repressing,11,1
1851,ZNF331,repressing,11,1


In [37]:
# ZNF385D
set(modules2[25].genes).intersection(set(mbvDEGs)) #fucking dope

{'DLX6-AS1',
 'GAD1',
 'GAD2',
 'LGI2',
 'PNOC',
 'PVALB',
 'SLC32A1',
 'SLC6A1',
 'ZNF385D'}

In [38]:
# PGAM2
set(modules2[13].genes).intersection(set(mbvDEGs)) #Astrocytes

{'ADM',
 'ANGPTL4',
 'APLNR',
 'AQP1',
 'CD44',
 'CHI3L1',
 'GBP1',
 'MAFF',
 'OSMR',
 'PGAM2',
 'SERPINA3',
 'VEGFA'}

In [35]:
# UGP2
set(modules2[21].genes).intersection(set(mbvDEGs)) #what I expected

{'APC',
 'ATP5F1E',
 'ATP5PF',
 'COX7A2',
 'COX7B',
 'HSPA8',
 'NDUFA5',
 'PRDX5',
 'SNHG14',
 'TOMM7',
 'UGP2'}

In [ ]:

    LOGGER.info("Loading databases.")
    dbs = _load_dbs(args.database_fname)